In [2]:
import numpy as np
from matplotlib import pyplot as plt
import math
import json
import subprocess
import time

config = {}

Lx = 0.1
Ly = 0.1
Lz = 0.1
nx = 100
ny = 200
nz = 1

config["mesh"] = {
    "type": "voxel",
    "Lx": Lx,
    "Ly": Ly,
    "Lz": Lz,
    "nx": nx,
    "ny": ny,
    "nz": nz
}

ncells = nx * ny * nz

rate = np.zeros(ncells, dtype=np.float64)
for i in range(ncells):
    rate[i] = 1.0e9

bgk_stddev = np.zeros(ncells, dtype=np.float64)
for i in range(ncells):
    bgk_stddev[i] = 1.0e4

bgk_mean_x = np.zeros(ncells, dtype=np.float64)
for i in range(ncells):
    bgk_mean_x[i] = 0.1

bgk_mean_y = np.zeros(ncells, dtype=np.float64)
bgk_mean_z = np.zeros(ncells, dtype=np.float64)

config["bgk_processes"] = [
    {
        "rate": rate.tolist(),
        "bgk_stddev": bgk_stddev.tolist(),
        "bgk_mean_x": bgk_mean_x.tolist(),
        "bgk_mean_y": bgk_mean_y.tolist(),
        "bgk_mean_z": bgk_mean_z.tolist()
    }
]

config["sources"] = [
    {
        "type": "point",
        "pos": [0, 0.05, 0.05],
        "vel_mean": [100, 0, 0],
        "vel_stddev": 100,
        "n_particles": 1000,
        "intensity": 1.0e10,
        "angle_min": -math.pi/2,
        "angle_max": math.pi/2,
    }
]

config["boundary_conditions"] = [
    {
        "type": "specular"
    },
    {
        "type": "specular"
    },
    {
        "type": "specular"
    },
    {
        "type": "partially_specular",
        "absorption_coefficient": 0.95
    },
    {
        "type": "specular"
    },
    {
        "type": "specular"
    },
]

config["max_steps"] = 1000000
config["seed"] = 0

config["t_final"] = 1.0e-4

inputstring = json.dumps(config)

with open("input.json", "w") as f:
    f.write(inputstring)

start_time = time.time()
subprocess.check_output(["neptune-mc"]).decode("utf-8")
end_time = time.time()

print("Simulation took", end_time - start_time, "seconds")

with open("output.json", "r") as f:
    outputstring = f.read()

output = json.loads(outputstring)

stats = output["stats"]
print(stats)

neutral_density = np.array(output["density"])

neutral_density_reshaped = np.reshape(neutral_density, (nx, ny), order='F')

c = plt.imshow(np.transpose(neutral_density_reshaped), extent=[0, Lx, 0, Ly])
plt.title("Neutral particle density")
plt.colorbar(c)
plt.xlabel("x")
plt.ylabel("y")
plt.show()


CalledProcessError: Command '['neptune-mc']' died with <Signals.SIGSEGV: 11>.